# Milestone 3 Pushshift Reddit Preprocessing & First Model Building and Evaluation

This notebook performs preprocessing and preliminary model building and evaluation on the Pushshift Reddit dataset.

In [1]:
# Dependencies 

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import requests

Matplotlib created a temporary cache directory at /scratch/ekim18/job_48803909/matplotlib-c0t2kvte because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# SparkSession Configuration

# 16 cores, 128GB total memory — local[*] mode
# In local mode there are no separate executor processes — all task execution
# runs as threads within a single JVM. Executor config parameters have no effect
# and are omitted. Driver memory is set to 120GB to give the JVM nearly the
# full node allocation.
spark = SparkSession.builder \
    .appName("PushshiftRedditPreprocessing") \
    .config("spark.driver.memory", "120g") \
    .config("spark.driver.maxResultSize", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark version: 3.5.0
Spark UI: http://exp-6-03.expanse.sdsc.edu:4040


In [3]:
# Data Load 

DATA_DIR = "../data/raw/"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))

COLS = ["author", "created_utc", "id", "num_comments", "score",
        "selftext", "subreddit", "subreddit_id", "title"]

pre_cutoff = [f for f in files if os.path.basename(f) >= "RS_2015-01"]
post_cutoff = [f for f in files if os.path.basename(f) < "RS_2015-01"]

# Files where created_utc is STRING - need to cast
df_pre = spark.read.parquet(*pre_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

df_post = spark.read.parquet(*post_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

MIN_TS = 1119398400  # June 2005
MAX_TS = 1700000000  # Nov 2023

df = df_pre.union(df_post) \
    .filter(F.col('created_utc').between(MIN_TS, MAX_TS))

print(f"Files loaded: {len(files)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")

Files loaded: 218
Partitions: 683


In [4]:
# SparkUI Screenshot

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
sparkUI_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
sparkUI_df['maxMemory_GB'] = (sparkUI_df['maxMemory'] / (1024**3)).round(2)
print(sparkUI_df)

# Spark context master check
print(spark.sparkContext.master)

       id  totalCores    maxMemory  activeTasks  isActive  maxMemory_GB
0  driver          16  77120667648            0      True         71.82
local[*]


In [5]:
# Hardcode row_count to dataset size for count verifications

row_count = 549662955

In [6]:
# ── FILTERING & CLEANING ─────────────────────────────────────────────────────

df_clean = df \
    .filter(F.col("subreddit").isNotNull()) \
    .filter(F.col("subreddit_id").isNotNull()) \
    .filter(F.col("score").isNotNull()) \
    .filter(F.col("num_comments") >= 0)

print(f"Original row count:      {row_count:,}")
clean_count = df_clean.count()
print(f"Row count after filters: {clean_count:,}")
print(f"Rows removed:            {row_count - clean_count:,}")

Original row count:      549,662,955
Row count after filters: 549,355,365
Rows removed:            307,590


## Bot Author Handling

Rather than attempting frequency-based bot detection (which would require a full groupBy on author + date across 549M rows to compute per-author daily post rates), we take a simpler and more interpretable approach: flagging known high-volume bot accounts identified in EDA as a binary feature `is_known_bot`.

This preserves all bot-authored posts in the dataset — bot posts have real and consistent engagement patterns (clustering heavily in low-engagement) that the model can learn from. Filtering them out would discard genuine signal. The `is_known_bot` flag gives the model explicit information about author type without requiring expensive per-author temporal aggregations.

The known bot list is seeded from the top authors output in EDA. Frequency-based detection can be revisited in as feature engineering improvement.

In [7]:
# ── BOT AUTHOR FLAGGING ───────────────────────────────────────────────────────

KNOWN_BOTS = {
    "AutoModerator", "AutoNewsAdmin", "AutoNewspaperAdmin",
    "politicbot", "RPBot", "ImagesOfNetwork", "-en-",
    "KellyfromLeedsUK"
}

df_clean = df_clean \
    .withColumn("is_known_bot",
        F.when(F.col("author").isin(list(KNOWN_BOTS)), 1)
         .otherwise(0)) \
    .withColumn("is_anonymous_author",
        F.when(
            (F.col("author") == "[deleted]") | (F.col("author") == ""), 1)
         .otherwise(0)) \
    .withColumn("has_title",
        F.when(F.length(F.col("title")) == 0, 0)
         .otherwise(1))

print("Author type features added.")
df_clean.select("is_known_bot", "is_anonymous_author", "has_title").show(5)

Author type features added.
+------------+-------------------+---------+
|is_known_bot|is_anonymous_author|has_title|
+------------+-------------------+---------+
|           0|                  1|        1|
|           0|                  1|        1|
|           0|                  0|        1|
|           0|                  1|        1|
|           0|                  0|        1|
+------------+-------------------+---------+
only showing top 5 rows



#### Distribution of subreddit post counts

First, we need to understand the shape of the subreddit post count distributions. 

In [8]:
subreddit_post_counts = df_clean \
    .groupBy("subreddit") \
    .count() \
    .withColumnRenamed("count", "post_count")

subreddit_post_counts.select("post_count").describe().show()

+-------+------------------+
|summary|        post_count|
+-------+------------------+
|  count|           2291802|
|   mean|239.70454908408317|
| stddev|16399.513624659332|
|    min|                 1|
|    max|          14521450|
+-------+------------------+



And the distribution at the low end specifically.

In [9]:
subreddit_post_counts.groupBy(
    F.when(F.col("post_count") < 10, "<10")
     .when(F.col("post_count") < 50, "10-49")
     .when(F.col("post_count") < 100, "50-99")
     .when(F.col("post_count") < 500, "100-499")
     .when(F.col("post_count") < 1000, "500-999")
     .otherwise("1000+")
     .alias("post_count_bucket")
) \
.count() \
.orderBy("post_count_bucket") \
.show()

+-----------------+-------+
|post_count_bucket|  count|
+-----------------+-------+
|            10-49| 291866|
|          100-499|  62694|
|            1000+|  26624|
|            50-99|  51649|
|          500-999|  13421|
|              <10|1845548|
+-----------------+-------+



### What percentage of total posts would be excluded at various thresholds?

In [10]:
for threshold in [10, 50, 100, 500, 1000]:
    kept = subreddit_post_counts.filter(F.col("post_count") >= threshold)
    n_subreddits = kept.count()
    n_posts = kept.agg(F.sum("post_count")).collect()[0][0]
    print(f"Threshold {threshold:5d}: {n_subreddits:8,} subreddits kept, {n_posts:12,} posts kept ({n_posts/549355364*100:.1f}%)")

Threshold    10:  446,254 subreddits kept,  545,144,030 posts kept (99.2%)
Threshold    50:  154,388 subreddits kept,  539,079,288 posts kept (98.1%)
Threshold   100:  102,739 subreddits kept,  535,480,818 posts kept (97.5%)
Threshold   500:   40,045 subreddits kept,  521,649,828 posts kept (95.0%)
Threshold  1000:   26,624 subreddits kept,  512,178,009 posts kept (93.2%)


## Subreddit Minimum Post Count Filter

Per-subreddit percentile thresholds are only statistically meaningful when a subreddit
has enough posts for `percentile_approx` to produce a reliable estimate. Subreddits
with very few posts produce degenerate thresholds — for example, a subreddit with 5
posts all scoring 1 has a median of 1, meaning any post with score ≥ 1 clears the
"high score" threshold, which is essentially every post on Reddit.

Analysis of the subreddit post count distribution reveals that 1,845,548 subreddits
(80.5% of all 2,291,802 subreddits) have fewer than 10 posts, and the distribution is
extremely long-tailed (mean: 239, stddev: 16,399). The vast majority of actual Reddit
activity is concentrated in a small number of active communities.

Post retention at various minimum thresholds:

| Min Posts | Subreddits Kept | Posts Kept |
|---|---|---|
| 10 | 446,254 | 99.2% |
| 50 | 154,388 | 98.1% |
| 100 | 102,739 | 97.5% |
| 500 | 40,045 | 95.0% |
| 1000 | 26,624 | 93.2% |

We select a minimum of **100 posts per subreddit** as the threshold. At this cutoff,
`percentile_approx` has sufficient data to produce stable estimates, the jump from 50
to 100 costs only 0.6% of posts while eliminating 51,649 additional unreliable
subreddits, and 97.5% post retention ensures the dataset remains representative.
Subreddits below this threshold are excluded from label generation entirely.

In [11]:
# ── SUBREDDIT MINIMUM POST COUNT FILTER ──────────────────────────────────────

MIN_SUBREDDIT_POSTS = 100

# Compute per-subreddit post counts and filter to active subreddits
active_subreddits = df_clean \
    .groupBy("subreddit") \
    .count() \
    .filter(F.col("count") >= MIN_SUBREDDIT_POSTS) \
    .select("subreddit")

df_active = df_clean.join(active_subreddits, on="subreddit", how="inner")

active_count = df_active.count()
print(f"Rows after subreddit filter: {active_count:,}")
print(f"Rows removed:                {clean_count - active_count:,}")
print(f"Posts retained:              {active_count / clean_count * 100:.1f}%")

Rows after subreddit filter: 535,480,818
Rows removed:                13,874,547
Posts retained:              97.5%


## Label Generation — Threshold Sensitivity Analysis

Before committing to a percentile threshold for engagement archetype assignment, we
evaluated how the class distribution shifts across a range of thresholds. The goal is
to find a threshold that produces a label distribution that reflects the reality of
Reddit engagement — truly viral posts should be rare, and low-engagement posts should
be the majority.

We ruled out standard deviation-based thresholds despite their statistical appeal
because Reddit score distributions are heavily right-skewed (global stddev: 707,
mean: 44.8), not normally distributed. Standard deviation thresholds assume roughly
normal distributions and behave unpredictably on skewed data. Percentile-based
thresholds are distribution-agnostic and more robust for this use case.

Sensitivity analysis was run exploratorily across percentiles 0.5 through 0.6 before
hitting compute constraints. Results:

| Percentile | viral | crowd-pleaser | debate-starter | low-engagement |
|---|---|---|---|---|
| 0.50 | 296,939,301 (55%) | 81,648,029 (15%) | 78,517,491 (15%) | 78,375,997 (15%) |
| 0.60 | 237,653,516 (44%) | 90,612,364 (17%) | 74,714,210 (14%) | 132,500,728 (25%) |

At both thresholds the viral class is the dominant class — the opposite of expected.
The trend shows viral dropping ~11% and low-engagement growing ~10% per 0.1 increment.
We select **0.75** as the final threshold, extrapolating that it brings the distribution
to a more intuitive shape where low-engagement is the plurality class. The 0.75
threshold is also conceptually clean — a post must beat 3 out of 4 posts in its
subreddit on both axes simultaneously to qualify as viral.

In [12]:
# ── LABEL GENERATION — 75TH PERCENTILE THRESHOLD ─────────────────────────────

SCORE_PERCENTILE = 0.75
COMMENTS_PERCENTILE = 0.75

subreddit_thresholds = df_active \
    .groupBy("subreddit") \
    .agg(
        F.expr(f"percentile_approx(score, {SCORE_PERCENTILE})").alias("score_thresh"),
        F.expr(f"percentile_approx(num_comments, {COMMENTS_PERCENTILE})").alias("comments_thresh")
    )

df_labeled = df_active.join(subreddit_thresholds, on="subreddit", how="left") \
    .withColumn("label_4class",
        F.when(
            (F.col("score") >= F.col("score_thresh")) &
            (F.col("num_comments") >= F.col("comments_thresh")), "viral"
        ).when(
            (F.col("score") >= F.col("score_thresh")) &
            (F.col("num_comments") < F.col("comments_thresh")), "crowd-pleaser"
        ).when(
            (F.col("score") < F.col("score_thresh")) &
            (F.col("num_comments") >= F.col("comments_thresh")), "debate-starter"
        ).otherwise("low-engagement")) \
    .withColumn("label_binary",
        F.when(
            (F.col("score") >= F.col("score_thresh")) |
            (F.col("num_comments") >= F.col("comments_thresh")),
            "high-engagement"
        ).otherwise("low-engagement"))

print("4-class label distribution:")
df_labeled.groupBy("label_4class") \
    .count() \
    .withColumn("pct", F.round(F.col("count") / F.lit(active_count) * 100, 1)) \
    .orderBy(F.col("count").desc()) \
    .show()

print("Binary label distribution:")
df_labeled.groupBy("label_binary") \
    .count() \
    .withColumn("pct", F.round(F.col("count") / F.lit(active_count) * 100, 1)) \
    .orderBy(F.col("count").desc()) \
    .show()

4-class label distribution:
+--------------+---------+----+
|  label_4class|    count| pct|
+--------------+---------+----+
|low-engagement|236954268|44.3|
|         viral|160168118|29.9|
| crowd-pleaser| 74916114|14.0|
|debate-starter| 63442318|11.8|
+--------------+---------+----+

Binary label distribution:
+---------------+---------+----+
|   label_binary|    count| pct|
+---------------+---------+----+
|high-engagement|298526550|55.7|
| low-engagement|236954268|44.3|
+---------------+---------+----+

